# 10 Cleanup Stale Paths (очистка отживших путей)

Этот notebook (ноутбук) чистит тяжелые `parquet paths (пути паркет-файлов)` на диске `D (диск D)`, но сохраняет:

- 236 стратегий из `fine tuning (тонкой настройки)`;
- 428 `self_sufficient_core (самодостаточных базовых стратегий)`;
- все стратегии, попавшие в `test portfolio (тестовый портфель)`.

По умолчанию запускается только `dry run (пробный запуск)` без удаления.

In [1]:
from pathlib import Path
import sys
import pandas as pd

WORK_ROOT = Path(r'./workspace')
if str(WORK_ROOT) not in sys.path:
    sys.path.insert(0, str(WORK_ROOT))

from cleanup_stale_paths import run_cleanup

manifest = run_cleanup(include_old_best_paths=True, execute=False)
manifest

{'keep_spec_ids': 664,
 'delete_files': 1542,
 'delete_size_gb': 3.99,
 'core_paths_dir': 'D:\\Filter_2_option_v2\\01_core_strategy_factory_output\\paths_selected',
 'test_paths_dir': 'D:\\Filter_2_option_v2\\08_test_run_output\\paths',
 'execute': False,
 'include_old_best_paths': True,
 'deleted_files': 0,
 'deleted_size_gb': 0.0,
 'failed_deletes': 0,
 'output_dir': 'D:\\Filter_2_option_v2\\10_cleanup_stale_paths_output',
 'saved_files': ['01_core_paths_audit.csv',
  '02_delete_plan.csv',
  '03_keep_spec_ids.csv',
  '04_deleted_files.csv',
  '05_delete_failures.csv',
  '06_manifest.json']}

## Delete Plan (план удаления)

Проверь эту таблицу перед реальным удалением.

In [2]:
OUTPUT_DIR = Path(r'D:\Filter_2_option_v2\10_cleanup_stale_paths_output')
plan = pd.read_csv(OUTPUT_DIR / '02_delete_plan.csv')
summary = plan.groupby('category').agg(files=('path', 'count'), size_gb=('size_bytes', lambda x: x.sum() / 1024**3)).reset_index()
summary

,category,files,size_gb
0,04_two_option_calibration_output_best_paths,570,0.573112
1,core_strategy_path,972,3.416674


In [3]:
plan.head(30)

,category,spec_id,strategy_id,path,exists,size_bytes,keep,delete_candidate
0,core_strategy_path,0.0,S6__time_series_momentum__lb_22,D:\Filter_2_option_v2\01_core_strategy_factory...,True,5045237,False,True
1,core_strategy_path,1.0,S6__time_series_reversal__lb_22,D:\Filter_2_option_v2\01_core_strategy_factory...,True,5128529,False,True
2,core_strategy_path,2.0,S6__time_series_momentum__lb_44,D:\Filter_2_option_v2\01_core_strategy_factory...,True,5009452,False,True
3,core_strategy_path,3.0,S6__time_series_reversal__lb_44,D:\Filter_2_option_v2\01_core_strategy_factory...,True,5088959,False,True
4,core_strategy_path,4.0,S6__time_series_momentum__lb_88,D:\Filter_2_option_v2\01_core_strategy_factory...,True,5024932,False,True
5,core_strategy_path,5.0,S6__time_series_reversal__lb_88,D:\Filter_2_option_v2\01_core_strategy_factory...,True,5048726,False,True
6,core_strategy_path,6.0,S6__time_series_momentum__lb_176,D:\Filter_2_option_v2\01_core_strategy_factory...,True,4997793,False,True
7,core_strategy_path,8.0,S6__time_series_momentum__lb_352,D:\Filter_2_option_v2\01_core_strategy_factory...,True,4878323,False,True
8,core_strategy_path,9.0,S6__time_series_reversal__lb_352,D:\Filter_2_option_v2\01_core_strategy_factory...,True,4904434,False,True
9,core_strategy_path,10.0,S6__ma_cross__fast_22__slow_88,D:\Filter_2_option_v2\01_core_strategy_factory...,True,4996098,False,True


## Execute Delete (реальное удаление)

Запускай эту ячейку только если `Delete Plan (план удаления)` выглядит правильно. Для удаления нужно вручную заменить `False` на `True`.

In [5]:
EXECUTE_DELETE = True

if not EXECUTE_DELETE:
    raise RuntimeError('Deletion is disabled. Set EXECUTE_DELETE = True only after checking the delete plan.')

manifest = run_cleanup(include_old_best_paths=True, execute=True)
manifest

{'keep_spec_ids': 664,
 'delete_files': 1542,
 'delete_size_gb': 3.99,
 'core_paths_dir': 'D:\\Filter_2_option_v2\\01_core_strategy_factory_output\\paths_selected',
 'test_paths_dir': 'D:\\Filter_2_option_v2\\08_test_run_output\\paths',
 'execute': True,
 'include_old_best_paths': True,
 'deleted_files': 1542,
 'deleted_size_gb': 3.99,
 'failed_deletes': 0,
 'output_dir': 'D:\\Filter_2_option_v2\\10_cleanup_stale_paths_output',
 'saved_files': ['01_core_paths_audit.csv',
  '02_delete_plan.csv',
  '03_keep_spec_ids.csv',
  '04_deleted_files.csv',
  '05_delete_failures.csv',
  '06_manifest.json']}

## Delete Report (отчет об удалении)

In [6]:
deleted = pd.read_csv(OUTPUT_DIR / '04_deleted_files.csv')
failures = pd.read_csv(OUTPUT_DIR / '05_delete_failures.csv')
display(deleted.groupby(deleted['path'].str.extract(r'Filter_2_option_v2\\([^\\]+)', expand=False)).agg(files=('path', 'count'), size_gb=('size_bytes', lambda x: x.sum() / 1024**3)))
display(failures)

,files,size_gb
path,,
01_core_strategy_factory_output,972,3.416674
04_two_option_calibration_output,570,0.573112


,path,error
